In [ ]:
!pip install xgboost 

In [1]:


from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from xgboost import XGBClassifier
from sklearn.svm import SVC
import pandas as pd
from data_preprocessing_module import preprocess_text  # Assuming preprocess_text is defined in this module

# Load and preprocess dataset
df = pd.read_csv('datasets/changed/data_news_1.csv')
df['label'] = df['type_of_news'].map({'Fake': 0, 'Real': 1})
df['title_clean'] = df['title'].apply(preprocess_text)

X = df['title_clean']
y = df['label']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TF-IDF vectorizer
tfidf = TfidfVectorizer(stop_words='english', max_df=0.7)

# Hyperparameter tuning for Random Forest
rf_pipeline = make_pipeline(tfidf, RandomForestClassifier(random_state=42))
rf_params = {
    'randomforestclassifier__n_estimators': [100, 200],
    'randomforestclassifier__max_depth': [None, 10, 20]
}
rf_grid = GridSearchCV(rf_pipeline, rf_params, cv=3, scoring='f1', n_jobs=-1)
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_

# Define models
models = {
    'Naive Bayes': make_pipeline(tfidf, MultinomialNB()),
    'Logistic Regression': make_pipeline(tfidf, LogisticRegression(max_iter=300)),
    'Random Forest (Tuned)': best_rf,
    'Gradient Boosting': make_pipeline(tfidf, GradientBoostingClassifier(random_state=42)),
    'XGBoost': make_pipeline(tfidf, XGBClassifier(use_label_encoder=False, eval_metric='logloss')),
    'SVM': make_pipeline(tfidf, SVC(probability=True, random_state=42))
}

# Evaluate models
model_average_scores = {}

for model_name, model in models.items():
    print(f"\nEvaluating {model_name}:")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    model_average_scores[model_name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    }

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))


print("\nModel Accuracy Comparison:")
for model_name, scores in model_average_scores.items():
    print(f"{model_name}: Accuracy: {scores['Accuracy']:.4f}")


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/kushalpanthi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/kushalpanthi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!



Evaluating Naive Bayes:
Accuracy: 0.7297
Precision: 0.8000
Recall: 0.6316
F1 Score: 0.7059

Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.83      0.75        18
           1       0.80      0.63      0.71        19

    accuracy                           0.73        37
   macro avg       0.74      0.73      0.73        37
weighted avg       0.74      0.73      0.73        37


Evaluating Logistic Regression:
Accuracy: 0.6216
Precision: 0.6471
Recall: 0.5789
F1 Score: 0.6111

Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.67      0.63        18
           1       0.65      0.58      0.61        19

    accuracy                           0.62        37
   macro avg       0.62      0.62      0.62        37
weighted avg       0.62      0.62      0.62        37


Evaluating Random Forest (Tuned):
Accuracy: 0.6216
Precision: 0.6087
Recall: 0.7368
F1 Score: 0.6667

Cla

/opt/anaconda3/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [23:20:16] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
